# 🎵 Spectral Agent - Standalone Training Notebook (v2.1)
This notebook is fully independent and contains all necessary code for downloading data, extracting features, training the CNN+LSTM+Attention model, and saving the results.

In [ ]:
!pip install -q torch torchaudio librosa soundfile scikit-learn matplotlib tqdm kaggle

In [ ]:
import os
import sys
import time
import json
import random
import subprocess
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import librosa
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve, auc as sklearn_auc, accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Configuration & Download
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = "data/asvspoof5"
KAGGLE_DATASET = "aniket202411001/asvspoof5-flac"
RESULTS_DIR = Path("results")
DATA_FACTOR = 0.6
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
PATIENCE = 5

def auto_download_dataset(data_dir, dataset_name):
    data_path = Path(data_dir)
    if data_path.exists() and any(data_path.iterdir()):
        print(f"[Dataset] Found at '{data_dir}'.")
        return
    print(f"[Dataset] Downloading from Kaggle into '{data_dir}'...")
    data_path.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download", "-d", dataset_name, "--unzip", "-p", str(data_path)], check=True)

# Uncomment and set if needed:
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'
try:
    auto_download_dataset(DATA_DIR, KAGGLE_DATASET)
except Exception as e:
    print(f"Warning: Could not download dataset: {e}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Audio & Feature Utilities
# ─────────────────────────────────────────────────────────────────────────────

def load_audio(path, target_sr=16000, mono=True):
    waveform, sr = librosa.load(str(path), sr=target_sr, mono=mono)
    peak = np.max(np.abs(waveform))
    if peak > 0: waveform = waveform / peak
    return waveform.astype(np.float32), target_sr

def pad_or_trim(waveform, target_length):
    n = len(waveform)
    if n == target_length: return waveform
    if n > target_length: return waveform[:target_length]
    return np.pad(waveform, (0, target_length - n), mode="constant")

def chunk_audio(waveform, sr=16000, chunk_duration=3.0, overlap=0.5):
    chunk_len = int(chunk_duration * sr)
    hop_len = int((chunk_duration - overlap) * sr)
    if len(waveform) <= chunk_len: return [pad_or_trim(waveform, chunk_len)]
    chunks = []
    start = 0
    while start + chunk_len <= len(waveform):
        chunks.append(waveform[start: start + chunk_len].copy())
        start += hop_len
    tail = waveform[start:]
    if len(tail) > chunk_len // 2: chunks.append(pad_or_trim(tail, chunk_len))
    return chunks

def extract_spectral_features(chunk, sr=16000):
    # Mel
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=chunk, sr=sr, n_mels=40, n_fft=512, hop_length=160), ref=np.max)
    # MFCC
    mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=20, n_fft=512, hop_length=160)
    # LFCC
    S = np.abs(librosa.stft(chunk, n_fft=512, hop_length=160))
    freq_points = np.linspace(0, S.shape[0] - 1, 72).astype(int)
    fb = np.zeros((70, S.shape[0]), dtype=np.float32)
    for m in range(1, 71):
        l, c, r = freq_points[m-1], freq_points[m], freq_points[m+1]
        for k in range(l, c + 1): 
            if c != l: fb[m-1, k] = (k - l) / (c - l)
        for k in range(c, r + 1): 
            if r != c: fb[m-1, k] = (r - k) / (r - c)
    lfcc = np.dot(np.cos(np.pi / 70 * (np.arange(20)[:, None] + 0.5) * np.arange(70)[None, :]), np.log(np.dot(fb, S) + 1e-8))
    # CQCC
    cqt = np.abs(librosa.cqt(chunk, sr=sr, hop_length=160, n_bins=84))
    cqcc = np.dot(np.cos(np.pi / 84 * (np.arange(20)[:, None] + 0.5) * np.arange(84)[None, :]), np.log(cqt + 1e-8))
    
    T = min(mel.shape[1], mfcc.shape[1], lfcc.shape[1], cqcc.shape[1])
    feats = np.concatenate([mel[:, :T], mfcc[:, :T], lfcc[:, :T], cqcc[:, :T]], axis=0)
    return ((feats - feats.mean(axis=1, keepdims=True)) / (feats.std(axis=1, keepdims=True) + 1e-8)).astype(np.float32)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Dataset & Dataloader
# ─────────────────────────────────────────────────────────────────────────────

class ASVspoofDataset(Dataset):
    def __init__(self, root_dir, split="train", data_factor=0.6, sr=16000):
        self.root = Path(root_dir)
        self.sr = sr
        self.items = []
        
        split_dir = self.root / split
        if not split_dir.exists():
            for c in ["flac_T", "flac_D", "flac_E_eval"]:
                if (self.root / c).exists() and split in c.lower():
                    split_dir = self.root / c; break
        
        bona_dir = split_dir / "bonafide"
        spoof_dir = split_dir / "spoof"
        
        bona_files = list(bona_dir.glob("**/*.flac")) + list(bona_dir.glob("**/*.wav"))
        spoof_files = list(spoof_dir.glob("**/*.flac")) + list(spoof_dir.glob("**/*.wav"))

        print(f"[Dataset] Found {len(bona_files)} bona and {len(spoof_files)} spoof files for {split}")
        
        # VERSION 2.1 FIX: Ensure we don't sample from empty lists
        if len(bona_files) > 0:
            k_bona = max(1, int(len(bona_files) * data_factor))
            bona_files = random.sample(bona_files, min(len(bona_files), k_bona))
        else:
            bona_files = []
            
        if len(spoof_files) > 0:
            k_spoof = max(1, int(len(spoof_files) * data_factor))
            spoof_files = random.sample(spoof_files, min(len(spoof_files), k_spoof))
        else:
            spoof_files = []
        
        for f in bona_files: self._expand(f, 0)
        for f in spoof_files: self._expand(f, 1)
        
        print(f"Loaded {len(self.items)} chunks for {split}")

    def _expand(self, path, label):
        try:
            wav, _ = load_audio(path, self.sr)
            chunks = chunk_audio(wav, self.sr)
            for i in range(len(chunks)):
                self.items.append((path, i, label))
        except:
            pass

    def __len__(self): return len(self.items)
    
    def __getitem__(self, idx):
        path, c_idx, label = self.items[idx]
        wav, _ = load_audio(path, self.sr)
        chunks = chunk_audio(wav, self.sr)
        feat = extract_spectral_features(chunks[c_idx], self.sr)
        return torch.from_numpy(feat), torch.tensor(label, dtype=torch.float32)

def collate_fn(batch):
    feats, labels = zip(*batch)
    transposed = [f.T for f in feats]
    padded = pad_sequence(transposed, batch_first=True).permute(0, 2, 1)
    return padded, torch.stack(labels)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Model Architecture (CNN + LSTM + Attention)
# ─────────────────────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(pool)
        )
    def forward(self, x): return self.net(x)

class AttentionPooling(nn.Module):
    def __init__(self, dim): 
        super().__init__()
        self.attn = nn.Linear(dim, 1)
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (w * x).sum(dim=1)

class SpectralModel(nn.Module):
    def __init__(self, n_features=100, lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128))
        self.proj = nn.Linear(max(1, n_features // 8) * 128, 128)
        self.lstm = nn.LSTM(128, lstm_hidden, num_layers=lstm_layers, batch_first=True, bidirectional=True)
        self.attn = AttentionPooling(lstm_hidden * 2)
        self.head = nn.Sequential(nn.Linear(lstm_hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        B, F, T = x.shape
        x = self.cnn(x.unsqueeze(1))
        B2, C, F2, T2 = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B2, T2, C * F2)
        x = F.relu(self.proj(x))
        x, _ = self.lstm(x)
        return torch.sigmoid(self.head(self.attn(x)).squeeze(-1))

    @torch.no_grad()
    def predict(self, feature_chunks):
        self.eval(); probs = []
        for c in feature_chunks:
            t = torch.from_numpy(c).unsqueeze(0).to(next(self.parameters()).device)
            probs.append(self.forward(t).item())
        return np.mean(probs)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Evaluation & Utilities
# ─────────────────────────────────────────────────────────────────────────────

def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    try:
        return brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except:
        return float(np.mean(np.abs(fnr - fpr)))

def save_results(dir_path, eer, auc, acc, f1):
    dir_path = Path(dir_path)
    dir_path.mkdir(parents=True, exist_ok=True)
    with open(dir_path / "results.txt", "w") as f:
        f.write(f"EER: {eer*100:.4f}%\nAUC: {auc:.4f}\nAccuracy: {acc*100:.4f}%\nF1: {f1:.4f}\n")
    with open(dir_path / "results.json", "w") as f:
        json.dump({"eer": eer, "auc": auc, "accuracy": acc, "f1": f1}, f, indent=4)
    print(f"Results saved to {dir_path}")

class Timer:
    def __init__(self, epochs, steps):
        self.epochs, self.steps, self.start = epochs, steps, time.time()
    def step(self, e, s, loss):
        elapsed = time.time() - self.start
        total_steps = self.epochs * self.steps
        current_step = e * self.steps + s
        eta = elapsed / max(1, current_step + 1) * (total_steps - current_step - 1)
        return f"\rEpoch {e+1}/{self.epochs} | Step {s+1}/{self.steps} | Loss: {loss:.4f} | ETA: {int(eta//60)}m {int(eta%60)}s   "


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Main Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device} (v2.1)")

    print("Loading datasets...")
    train_ds = ASVspoofDataset(DATA_DIR, "train", data_factor=DATA_FACTOR)
    val_ds = ASVspoofDataset(DATA_DIR, "val", data_factor=DATA_FACTOR)
    test_ds = ASVspoofDataset(DATA_DIR, "test", data_factor=DATA_FACTOR)
    
    if len(train_ds) == 0: 
        print("No training data found. Please check DATA_DIR.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = SpectralModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()
    
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    best_eer = float('inf')
    patience_ctr = 0
    
    timer = Timer(EPOCHS, len(train_loader))
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for i, (feats, labels) in enumerate(train_loader):
            feats, labels = feats.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(feats)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            print(timer.step(epoch, i, loss.item()), end="")
            
        print()
        
        # Validation
        model.eval()
        val_labels, val_scores = [], []
        with torch.no_grad():
            for feats, labels in val_loader:
                preds = model(feats.to(device))
                val_scores.extend(preds.cpu().numpy())
                val_labels.extend(labels.numpy())
                
        val_labels, val_scores = np.array(val_labels), np.array(val_scores)
        eer = compute_eer(val_labels, val_scores)
        print(f"Epoch {epoch+1} Validation EER: {eer*100:.2f}%")
        
        if eer < best_eer:
            best_eer = eer
            patience_ctr = 0
            torch.save({"model_state_dict": model.state_dict()}, RESULTS_DIR / "best.pt")
            print(" -> Saved new best model!")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print("Early stopping triggered.")
                break

    # Final Evaluation
    print("\nRunning final evaluation on Test Set...")
    model.load_state_dict(torch.load(RESULTS_DIR / "best.pt")["model_state_dict"])
    model.eval()
    test_labels, test_scores = [], []
    with torch.no_grad():
        for feats, labels in test_loader:
            preds = model(feats.to(device))
            test_scores.extend(preds.cpu().numpy())
            test_labels.extend(labels.numpy())
            
    test_labels, test_scores = np.array(test_labels), np.array(test_scores)
    test_preds = (test_scores >= 0.5).astype(int)
    
    eer = compute_eer(test_labels, test_scores)
    auc = sklearn_auc(*roc_curve(test_labels, test_scores, pos_label=1)[:2])
    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds)
    
    save_results(RESULTS_DIR, eer, auc, acc, f1)

if __name__ == "__main__":
    train()
